# 多变量时间序列预测

PipelineTS 支持多变量预测，适用于多个特征同时影响目标变量的场景。

支持三种模式：
1. **单变量 (Univariate)**: `target_col='y'`, `feature_cols=None` — 经典单变量预测
2. **多输入单输出 (Multi-input Single-output)**: `target_col='y'`, `feature_cols=['a','b','y']` — 多个特征预测单个目标
3. **多输入多输出 (Multi-input Multi-output)**: `target_col=['a','b']`, `feature_cols=['a','b','c']` — 多个特征预测多个目标

目前支持多变量的模型：**ITransformerModel**, **SRSNetModel**

In [ ]:
import numpy as np
import pandas as pd

# 准备多变量示例数据
np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
data = pd.DataFrame({
    'date': dates,
    'value': np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.1,
    'feature_a': np.cos(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.1,
    'feature_b': np.sin(np.linspace(0, 2 * np.pi, n)) * 0.5 + np.random.randn(n) * 0.05
})

LAGS = 12
PREDICT_N = 10

print(f"数据形状: {data.shape}")
data.head()

## 1. 单变量预测（基准）

不指定 `feature_cols`，模型仅使用目标列进行预测。

In [ ]:
from PipelineTS.nn_model import ITransformerModel

model_uni = ITransformerModel(
    time_col='date', target_col='value', lags=LAGS,
    d_model=32, n_heads=2, d_ff=64, e_layers=1,
    quantile=None, epochs=50, patience=10, verbose=False
)
model_uni.fit(data)
result_uni = model_uni.predict(PREDICT_N)
print("单变量预测结果:")
result_uni

## 2. 多输入单输出预测

指定 `feature_cols` 包含多个列，模型利用所有特征来预测单个目标变量。

In [ ]:
model_miso = ITransformerModel(
    time_col='date',
    target_col='value',
    feature_cols=['value', 'feature_a', 'feature_b'],
    lags=LAGS,
    d_model=32, n_heads=2, d_ff=64, e_layers=1,
    quantile=None, epochs=50, patience=10, verbose=False
)
model_miso.fit(data)
result_miso = model_miso.predict(PREDICT_N)
print("多输入单输出预测结果:")
result_miso

## 3. 使用 SRSNetModel 进行多变量预测

SRSNet 也支持多变量模式，使用多尺度自适应 patch 和选择性表征。

In [ ]:
from PipelineTS.nn_model import SRSNetModel

model_srs = SRSNetModel(
    time_col='date',
    target_col='value',
    feature_cols=['value', 'feature_a', 'feature_b'],
    lags=LAGS,
    d_model=32, n_heads=2,
    quantile=None, epochs=50, patience=10, verbose=False
)
model_srs.fit(data)
result_srs = model_srs.predict(PREDICT_N)
print("SRSNet 多变量预测结果:")
result_srs

## 4. 在 Pipeline 中使用多变量模型

ModelPipeline 也支持 `feature_cols` 参数，会自动传递给支持多变量的模型。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

pipeline = ModelPipeline(
    time_col='date',
    target_col='value',
    feature_cols=['value', 'feature_a', 'feature_b'],
    lags=LAGS,
    include_models=['lightgbm', 'xgboost'],  # ML 模型也可以在 pipeline 中
    quantile=None,
    cv=2
)

leaderboard = pipeline.fit(data)
print("Pipeline 排行榜:")
leaderboard

## 总结

| 模式 | target_col | feature_cols | 说明 |
|------|-----------|-------------|------|
| 单变量 | `'y'` | `None` | 经典单变量 |
| 多输入单输出 | `'y'` | `['a','b','y']` | 多特征→单目标 |
| 多输入多输出 | `['a','b']` | `['a','b','c']` | 多特征→多目标 |